# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library, referencing all schema resources by their Croissant `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset object
dataset = mlc.Dataset(croissant_url)
# View dataset metadata as a Python object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Let's review available record sets referenced by their `@id`, and list the fields and columns for each.

In [ ]:
# List available record sets and their fields in the Croissant schema
from pprint import pprint

record_sets = list(dataset.record_sets)
print(f"Available record set @ids: {[rs['@id'] for rs in record_sets]}")

# For each record set, print its @id, name, and list field (column) @ids
for rs in record_sets:
    print(f"\nRecord Set @id: {rs['@id']}")
    print(f"  name: {rs.get('name', '')}")
    if 'field' in rs:
        field_ids = [field['@id'] if isinstance(field, dict) else field for field in rs['field']]
        print(f"  Fields: {field_ids}")
    elif 'column' in rs:
        col_ids = [col['@id'] if isinstance(col, dict) else col for col in rs['column']]
        print(f"  Columns: {col_ids}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s found above.

In [ ]:
# Choose one record set for example analysis -- most datasets have a main record set for tabular data.

# (Update these after viewing the output of the previous cell!)
main_record_set_id = 'cr:RecordSet/mental_health_survey_responses'  # Replace with the correct @id shown above

extracted_record_sets = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}

for record_set_id in extracted_record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for record set '{record_set_id}' with shape {dataframes[record_set_id].shape}")
    else:
        print(f"No records found or unable to load records for record set: {record_set_id}")

# Show columns for the main record set chosen
if main_record_set_id in dataframes:
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print(f"Record set {main_record_set_id} was not loaded. Check the record set ids in the previous cell output.")

## 4. Exploratory Data Analysis (EDA)
Let's perform basic EDA for a numeric field (e.g., 'phq9_total_score' or an anxiety/depression score), filtering, normalization, and grouping. All operations refer to field/column names by their Croissant `@id` as provided in the schema.

In [ ]:
import numpy as np

# Reference a numeric field by its @id as defined in the schema (update as needed after viewing the DataFrame columns)
numeric_field_id = 'cr:Field/phq9_total_score'        # Example: PHQ-9 questionnaire total score
group_field_id = 'cr:Field/level_of_education'        # Example: Education level

# Which record set DataFrame to use
df = dataframes.get(main_record_set_id)

if df is not None and numeric_field_id in df.columns:
    # Filter for records with a score above a threshold
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} rows")
    display(filtered_df[[numeric_field_id]].head())

    # Normalize the numeric field
    filtered_df[f'{numeric_field_id}_normalized'] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head())

    # Group by an educational attribute, e.g., level of education
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Average {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
    else:
        print(f"Group field {group_field_id} does not exist in columns: {df.columns.tolist()}")
else:
    print(f"numeric_field_id ({numeric_field_id}) not found in columns: {df.columns.tolist()} if df is not None else 'No dataframe loaded'.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and display how it varies by category (e.g., by education level).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=16, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (PHQ-9 Score)")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot by education group if applicable
    if group_field_id in df.columns:
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to load and explore the FAIR² Mental Health Survey dataset using the `mlcroissant` library, referencing all data resources by their Croissant schema `@id`. We:
1. Loaded metadata and reviewed the available record sets and fields by `@id`.
2. Extracted data for analysis into pandas DataFrames.
3. Performed basic EDA, including numeric field filtering, normalization, and grouping by attributes.
4. Visualized field distributions and group comparisons.

For your own analyses, adapt the record set and field IDs as needed per your research focus. This notebook illustrates best practices for FAIR data exploration using open, schema-driven tools.